# 03 - Avanzado: rendimiento, memoria acotada, outliers

Este notebook asume los dos anteriores. Temas:

- Por qué filtrar por `PERIODO` es barato (poda de row groups por las
  estadísticas min/max de Parquet, gracias a que el histórico está ordenado
  por `PERIODO` primero).
- `polars` en modo *lazy* para no cargar todo a RAM (relevante si estás en
  una máquina chica; ver `README.md`, sección "Corrido en una máquina con
  4 GB de RAM").
- Procesar por partes (año por año) cuando una consulta no entra cómoda en
  memoria de una sola vez.
- Detección simple de outliers en `FOB_UNITARIO_USD` dentro de una NCM.
- Exportar un subconjunto grande a Parquet/Excel sin explotar la RAM.

In [ ]:
import time
import duckdb
import polars as pl

HISTORICO = "../Data/impo_historico.parquet"
con = duckdb.connect()
# Limite explicito, igual que el resto del pipeline (ver README, "Corrido en
# una máquina con 4 GB de RAM"): sin esto, DuckDB intenta usar la RAM libre
# que encuentre, y en una máquina chica una consulta pesada puede fallar por
# falta de memoria en vez de derramar a disco de forma controlada.
con.execute("PRAGMA memory_limit='1.5GB'")
con.execute("PRAGMA temp_directory='../Data'")  # sin esto, al tocar el limite la consulta falla en vez de derramar a disco
con.execute(f"CREATE VIEW impo AS SELECT * FROM read_parquet('{HISTORICO}')")


## Filtrar por fecha es (mucho) más barato que un `count(*)` sin filtro

El histórico está ordenado por `PERIODO, DESTINACION, NUM_ITEM,
ARANCEL_CONCEPTO` (ver README, "Por qué DuckDB y por qué Parquet"). Eso
significa que cada row group de Parquet cubre un rango chico y contiguo de
`PERIODO`, y DuckDB puede saltear enteros los row groups que no matchean un
filtro de `PERIODO` sin leerlos. Comparar los tiempos:

In [ ]:
t0 = time.perf_counter()
con.execute("SELECT count(*) FROM impo").fetchone()
t_full = time.perf_counter() - t0

t0 = time.perf_counter()
con.execute("SELECT count(*) FROM impo WHERE PERIODO = (SELECT max(PERIODO) FROM impo)").fetchone()
t_un_mes = time.perf_counter() - t0

print(f"count(*) sin filtro:      {t_full:.2f}s")
print(f"count(*) un solo PERIODO: {t_un_mes:.2f}s")


## `polars` lazy: mirar el plan antes de ejecutar

`pl.scan_parquet` arma un plan (`LazyFrame`) sin leer nada todavía.
`.explain()` muestra qué va a hacer *antes* de correrlo: si el filtro y la
selección de columnas quedan aplicados desde el scan (`PROJECT`/`FILTER`
empujados hacia abajo), significa que nunca va a materializar las columnas ni
las filas que no hacen falta.

El filtro toma los últimos 12 meses relativos al máximo `PERIODO` disponible
(no una fecha fija), para que el recorte se mantenga chico sin importar
cuánto haya crecido el histórico cuando corras esto:

In [ ]:
ultimo = con.execute("SELECT max(PERIODO) FROM impo").fetchone()[0]
desde = f"{int(ultimo[:4]) - 1}{ultimo[4:]}"
print(f"Ultimos 12 meses: {desde} a {ultimo}")

plan = (
    pl.scan_parquet(HISTORICO)
    .filter(pl.col("PERIODO") >= desde)
    .select(["PERIODO", "POS_NCM", "FOB_TOTAL_USD"])
)
print(plan.explain())


In [ ]:
resultado = plan.group_by("PERIODO").agg(pl.col("FOB_TOTAL_USD").sum()).collect()
resultado.sort("PERIODO")


## Procesar por partes cuando algo no entra en memoria

Si una consulta sobre el histórico completo no entra cómoda en la RAM
disponible (por ejemplo, un `GROUP BY` con muchas claves distintas sobre
todos los años), una salida simple es partir el trabajo por año y combinar
resultados ya agregados (mucho más chicos) al final, en vez de agregar todo
de una sola pasada.

Importante: el filtro de cada parte tiene que ser un rango directo sobre
`PERIODO` (`BETWEEN`), no `substr(PERIODO, 1, 4) = anio`. Envolver la columna
en una función le esconde a DuckDB el rango real, así que no puede podar row
groups por las estadísticas min/max de Parquet y termina escaneando el
archivo completo en cada una de las N vueltas del loop (N veces más lento, y
en una máquina chica, N veces más chances de quedarse sin memoria).

In [ ]:
anios = con.execute("SELECT DISTINCT substr(PERIODO,1,4) FROM impo ORDER BY 1").fetchall()
anios = [a[0] for a in anios]

partes = []
for anio in anios:
    df = con.execute(f'''
        SELECT POS_NCM, round(sum(FOB_TOTAL_USD)) AS fob_usd
        FROM impo
        WHERE PERIODO BETWEEN '{anio}01' AND '{anio}12'
        GROUP BY POS_NCM
    ''').fetchdf()
    df["anio"] = anio
    partes.append(df)

import pandas as pd
combinado = pd.concat(partes, ignore_index=True)
combinado.groupby("POS_NCM")["fob_usd"].sum().sort_values(ascending=False).head(20)


## Outliers de precio unitario dentro de una NCM

Regla simple basada en rango intercuartílico (IQR): un `FOB_UNITARIO_USD`
muy por fuera del rango típico de su propia NCM puede ser un error de carga,
una unidad de medida mal declarada, o un caso real que vale la pena mirar a
mano. No es una verdad absoluta, es un filtro para priorizar qué revisar.

In [ ]:
NCM_PREFIJO = "8504"
con.execute(f'''
    WITH items AS (
        SELECT DESTINACION, NUM_ITEM,
               any_value(POS_NCM) AS ncm,
               any_value(FOB_UNITARIO_USD) AS fob_unitario
        FROM impo
        WHERE POS_NCM LIKE '{NCM_PREFIJO}%' AND FOB_UNITARIO_USD IS NOT NULL
        GROUP BY DESTINACION, NUM_ITEM
    ),
    stats AS (
        SELECT
            quantile_cont(fob_unitario, 0.25) AS q1,
            quantile_cont(fob_unitario, 0.75) AS q3
        FROM items
    )
    SELECT i.*
    FROM items i, stats s
    WHERE i.fob_unitario > s.q3 + 1.5 * (s.q3 - s.q1)
       OR i.fob_unitario < s.q1 - 1.5 * (s.q3 - s.q1)
    ORDER BY i.fob_unitario DESC
    LIMIT 20
''').fetchdf()


## Exportar un subconjunto grande sin explotar la RAM

`COPY ... TO ... (FORMAT PARQUET)` corre del lado de DuckDB (no pasa por un
DataFrame de pandas en el medio), así que sirve para exportar subconjuntos
grandes sin duplicar la data en memoria de Python. Para recortes por lista de
NCM en particular, ver `filtrar_posiciones.py` en la raíz del repo (mismo
patrón, ya armado como script).

In [ ]:
con.execute('''
    COPY (
        SELECT * FROM impo WHERE POS_NCM LIKE '8504%'
    ) TO 'salida_ncm_8504.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
''')


## Dónde seguir

- `README.md`: arquitectura completa, esquema de columnas, y cómo se mide el
  ahorro de espacio de Parquet frente al `.lst` crudo.
- `docs/EXPLORACION_ONLINE.md` + `explorer/index.html`: prototipo para
  explorar el histórico desde el navegador, sin instalar nada, corriendo SQL
  contra el Parquet directo (mismo motor DuckDB, pero via DuckDB-WASM).
- `python Data/descargar_historico_impo.py --actualizar`: para traer el
  histórico al día antes de repetir este análisis con datos más recientes.